### Lading - Preparação e inspeção

In [0]:
import re
from datetime import datetime

# Padronização dos nomes dos schemas e volumes
CATALOG = "vcredit_medallion"
BRONZE_SCHEMA_NAME = "bronze"
SILVER_SCHEMA_NAME = "silver"
GOLD_SCHEMA_NAME = "gold"

BRONZE_SCHEMA = f"{CATALOG}.{BRONZE_SCHEMA_NAME}"
SILVER_SCHEMA = f"{CATALOG}.{SILVER_SCHEMA_NAME}"
GOLD_SCHEMA = f"{CATALOG}.{GOLD_SCHEMA_NAME}"

LANDING_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA_NAME}/landing"

# Símbolos padronizados de log
OK = "✅"
FAIL = "❌"
WARN = "⚠️"

# Para verificação visual
print(f"CATALOG: {CATALOG}")
print(f"BRONZE_SCHEMA: {BRONZE_SCHEMA}")
print(f"SILVER_SCHEMA: {SILVER_SCHEMA}")
print(f"GOLD_SCHEMA: {GOLD_SCHEMA}")
print(f"LANDING_PATH: {LANDING_PATH}")



CATALOG: vcredit_medallion
BRONZE_SCHEMA: vcredit_medallion.bronze
SILVER_SCHEMA: vcredit_medallion.silver
GOLD_SCHEMA: vcredit_medallion.gold
LANDING_PATH: /Volumes/vcredit_medallion/bronze/landing


In [0]:
# Cria, apenas se já não existem os schemas e volumes garantindo idempotência.
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {BRONZE_SCHEMA}.landing")

DataFrame[]

In [0]:
# Utiliza o módulo re (regex) para corrigir possíveis títulos errados antes de verificar a presença do arquivo.
# Cria o dicionário contento os nomes dos arquivos esperados bem como os respectivos regex para buscar nomes diferentes (contendo ano, por exemplo)
# e não diferenciando maiúsculas e minúsculas.
expected_files = {
    "base_atendentes.csv": r"(?i)base_atendentes.*\.csv$",
    "base_motivos.csv": r"(?i)base_motivos.*\.csv$",
    "canais.csv": r"(?i)canais.*\.csv$",
    "chamados.csv": r"(?i)chamados.*\.csv$",
    "chamados_hora.csv": r"(?i)Chamados_Hora.*\.csv$",
    "clientes.csv": r"(?i)clientes.*\.csv$",
    "custos.csv": r"(?i)custos.*\.csv$",
    "pesquisa_satisfacao.csv": r"(?i)pesquisa_satisfacao.*\.csv$",
}

# Verifica se todos os arquivos estão presentes e corrige o nome se necessário, removendo datas, por exemplo.
file_infos = dbutils.fs.ls(LANDING_PATH)
found_files = [f.name for f in file_infos]

empty_files = [f.name for f in file_infos if f.size == 0]
if empty_files:
    print(f"{FAIL} Arquivos vazios detectados: {empty_files}")
    raise Exception(f"Arquivos vazios encontrados: {empty_files}")

print("Encontrados:", found_files)

renamed_files = []
missing_files = []


for right_name, regex in expected_files.items():
    if right_name in found_files:
        continue

    wrong_name = None
    for i in found_files:
        if re.match(regex, i):
            wrong_name = i
            break

    if wrong_name:
        print(f"Renomeando {wrong_name} -> {right_name}")
        dbutils.fs.mv(f"{LANDING_PATH}/{wrong_name}", f"{LANDING_PATH}/{right_name}")
        found_files.remove(wrong_name)
        found_files.append(right_name)
        renamed_files.append((wrong_name, right_name))
    else:
        missing_files.append(right_name)

if missing_files:
    print(f"{FAIL} Arquivos faltando detectados")
    raise Exception(f"Arquivos faltando: {missing_files}")
else:
    print(f"{OK} Todos os arquivos estão presentes.")

Encontrados: ['base_atendentes.csv', 'base_motivos.csv', 'canais.csv', 'chamados.csv', 'chamados_hora.csv', 'clientes.csv', 'custos.csv', 'pesquisa_satisfacao.csv']
✅ Todos os arquivos estão presentes.


In [0]:
from functools import reduce
from pyspark.sql import functions as F

for i in expected_files:
    print(f"\n{'='*40}\n--- {i} ---")
    
    # Le como string (padrão do Bronze)
    df = spark.read.csv(f"{LANDING_PATH}/{i}", header=True, inferSchema=False)
    df.printSchema()
    
    # Mostra as primeiras linhas
    print("Amostra de dados:")
    display(df.limit(100))
    
    # Inspeção de assinatura em todas as colunas
    dfs_assinaturas = []
    
    for col_name in df.columns:
        df_sig = (
            df.select(
                F.lit(col_name).alias("coluna"),
                F.regexp_replace(
                    F.regexp_replace(
                        F.trim(F.lower(F.col(col_name))), 
                        r"\d", "9"
                    ),
                    r"[a-z]", "A"
                ).alias("assinatura")
            )
            .groupBy("coluna", "assinatura")
            .count()
        )
        dfs_assinaturas.append(df_sig)
    
    # Mostra o resultado de todas as colunas
    df_todas_assinaturas = reduce(lambda df1, df2: df1.unionByName(df2), dfs_assinaturas)
    
    print(f"Assinaturas de {i}:")
    display(
        df_todas_assinaturas.orderBy("coluna", F.col("count").desc()).limit(100)
    )


--- base_atendentes.csv ---
root
 |-- id_atendente: string (nullable = true)
 |-- nome_atendente: string (nullable = true)
 |-- nivel_atendimento: string (nullable = true)
 |-- data_admissao: string (nullable = true)
 |-- data_desligamento: string (nullable = true)
 |-- status_atendente: string (nullable = true)

Amostra de dados:


id_atendente,nome_atendente,nivel_atendimento,data_admissao,data_desligamento,status_atendente
1,Marcela França,1,18/04/2019,16/11/2025,inativo
2,Zilda Esteves,1,12/09/2023,24/01/2026,inativo
3,William Marques,1,28/05/2019,15/04/2025,inativo
4,Solange Silveira,Nível 1,07/10/2019,null,ativo
5,Joaquim Barros,Nível 1,06/12/2020,23/05/2026,inativo
6,Roberta Assunção,1,29/10/2021,null,ativo
7,Luiza Galvão,Nível 1,24/04/2022,null,ativo
8,Josefa Brito,Nível 1,25/01/2023,null,ativo
9,Jaqueline Campos,1,19/04/2019,null,ativo
10,Marcos Torres,1,20/01/2020,null,ativo


Assinaturas de base_atendentes.csv:


coluna,assinatura,count
data_admissao,99/99/9999,25
data_desligamento,null,20
data_desligamento,99/99/9999,5
id_atendente,99,16
id_atendente,9,9
nivel_atendimento,9,17
nivel_atendimento,AíAAA 9,7
nivel_atendimento,A9,1
nome_atendente,AAAAAAA AAAAAA,2
nome_atendente,AAAAAA AAAAA,2



--- base_motivos.csv ---
root
 |-- id_motivo: string (nullable = true)
 |-- nome_motivo: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- criticidade: string (nullable = true)

Amostra de dados:


...

coluna,assinatura,count
custo,"9,99",51053
custo,"99,99",27031
custo,9.99,16943
custo,"A$ 9,99",10211
custo,99.99,9167
custo,9.99AAAAA,6889
custo,"A$ 99,99",5380
custo,99.99AAAAA,3658
custo,"999,99",287
custo,"-9,99",98



--- pesquisa_satisfacao.csv ---
root
 |-- id_pesquisa: string (nullable = true)
 |-- id_chamado: string (nullable = true)
 |-- nota_atendimento: string (nullable = true)

Amostra de dados:


id_pesquisa,id_chamado,nota_atendimento
71619,115746,3
13534,19389,4
53852,83856,4
5563,7771,5
15959,22934,5
77687,127019,3
74949,121886,5
29267,43292,4
27051,39785,3
78680,128918,2


Assinaturas de pesquisa_satisfacao.csv:


coluna,assinatura,count
id_chamado,99999,56478
id_chamado,999999,17345
id_chamado,9999,6478
id_chamado,999,650
id_chamado,99,66
id_chamado,9,4
id_pesquisa,99999,71022
id_pesquisa,9999,9000
id_pesquisa,999,900
id_pesquisa,99,90
